# BreakHis — Breast Cancer Histopathology Classification
## CNN+ViT and EfficientNet-B4+ViT
### Binary (2-class) + Multiclass (8-class) — Combined Magnification Training

**Run cells in order — do not skip any cell.**

**4 models will be trained:**
| # | Model | Task | Classes |
|---|---|---|---|
| 1 | CNN+ViT | Binary | 2 (benign / malignant) |
| 2 | EfficientNet+ViT | Binary | 2 (benign / malignant) |
| 3 | CNN+ViT | Multiclass | 8 (tumor subtypes) |
| 4 | EfficientNet+ViT | Multiclass | 8 (tumor subtypes) |

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Step 2 — Paths (already set — do not change)

In [ ]:
import os

DRIVE_BASE              = "/content/drive/MyDrive/BreakHis dataset with model traning and testing"
BINARY_DATASET_PATH     = f"{DRIVE_BASE}/classificacao_binaria"
MULTICLASS_DATASET_PATH = f"{DRIVE_BASE}/classificacao_multiclasse"
CODE_PATH               = f"{DRIVE_BASE}/colab"
RESULTS_BINARY          = f"{DRIVE_BASE}/breakhis_results/binary"
RESULTS_MULTICLASS      = f"{DRIVE_BASE}/breakhis_results/multiclass"

os.makedirs(RESULTS_BINARY,     exist_ok=True)
os.makedirs(RESULTS_MULTICLASS, exist_ok=True)

print('Binary dataset   :', '✓ Found' if os.path.exists(BINARY_DATASET_PATH)     else '✗ NOT FOUND - check Drive')
print('Multiclass dataset:', '✓ Found' if os.path.exists(MULTICLASS_DATASET_PATH) else '✗ NOT FOUND - check Drive')
print('Code folder      :', '✓ Found' if os.path.exists(CODE_PATH)               else '✗ NOT FOUND - check Drive')
print('Results (binary) :', RESULTS_BINARY)
print('Results (multi)  :', RESULTS_MULTICLASS)

## Step 3 — Install Dependencies

In [ ]:
!pip install timm seaborn -q
print('✓ Dependencies installed.')

## Step 4 — Copy Dataset to Local Storage (faster training)
**This takes 5-10 minutes but makes each epoch 5x faster.**

In [ ]:
import shutil, os

print("Copying dataset to local storage (do this once per session)...")

if not os.path.exists("/content/classificacao_binaria"):
    print("  Copying binary dataset...")
    shutil.copytree(BINARY_DATASET_PATH, "/content/classificacao_binaria")
    print("  ✓ Binary dataset copied.")
else:
    print("  ✓ Binary dataset already copied.")

if not os.path.exists("/content/classificacao_multiclasse"):
    print("  Copying multiclass dataset...")
    shutil.copytree(MULTICLASS_DATASET_PATH, "/content/classificacao_multiclasse")
    print("  ✓ Multiclass dataset copied.")
else:
    print("  ✓ Multiclass dataset already copied.")

# Point scripts to local copy
os.environ["BREAKHIS_ROOT"]            = "/content/classificacao_binaria"
os.environ["BREAKHIS_MULTICLASS_ROOT"] = "/content/classificacao_multiclasse"
os.environ["RESULTS_DIR"]             = RESULTS_BINARY
os.environ["RESULTS_DIR_MULTICLASS"]  = RESULTS_MULTICLASS

print("\n✓ Ready to train!")

## Step 5 — Setup & GPU Check

In [ ]:
import sys
sys.path.insert(0, CODE_PATH)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
# PART A — Binary Classification (2 classes: benign / malignant)

## A1 — Train CNN+ViT (Binary)

In [ ]:
!cd "{CODE_PATH}" && python train/train_breakhis.py --model cnn_vit --epochs 50 --batch_size 32

## A2 — Evaluate CNN+ViT (Binary)

In [ ]:
!cd "{CODE_PATH}" && python evaluate/metrics.py --model cnn_vit

## A3 — Train EfficientNet+ViT (Binary)

In [ ]:
!cd "{CODE_PATH}" && python train/train_breakhis.py --model efficientnet_vit --epochs 50 --batch_size 32

## A4 — Evaluate EfficientNet+ViT (Binary)

In [ ]:
!cd "{CODE_PATH}" && python evaluate/metrics.py --model efficientnet_vit

---
# PART B — Multiclass Classification (8 tumor subtypes)

## B1 — Train CNN+ViT (Multiclass)

In [ ]:
!cd "{CODE_PATH}" && python train/train_breakhis_multiclass.py --model cnn_vit --epochs 50 --batch_size 32

## B2 — Evaluate CNN+ViT (Multiclass)

In [ ]:
!cd "{CODE_PATH}" && python evaluate/metrics_multiclass.py --model cnn_vit

## B3 — Train EfficientNet+ViT (Multiclass)

In [ ]:
!cd "{CODE_PATH}" && python train/train_breakhis_multiclass.py --model efficientnet_vit --epochs 50 --batch_size 32

## B4 — Evaluate EfficientNet+ViT (Multiclass)

In [ ]:
!cd "{CODE_PATH}" && python evaluate/metrics_multiclass.py --model efficientnet_vit

---
# Final Summary — All Results

In [ ]:
import json, os

print('='*60)
print('  BINARY CLASSIFICATION RESULTS (2 classes)')
print('='*60)
for model_name in ['cnn_vit', 'efficientnet_vit']:
    path = os.path.join(RESULTS_BINARY, f'{model_name}_metrics.json')
    if not os.path.exists(path):
        print(f'  {model_name}: not found'); continue
    with open(path) as f:
        m = json.load(f)
    o = m['overall']
    print(f'\n  {model_name.upper()}')
    print(f"  Overall  → Acc: {o['accuracy']*100:.2f}%  F1: {o['f1_macro']*100:.2f}%  AUC: {o['auc_roc']*100:.2f}%")
    for mag, mm in m.get('per_magnification', {}).items():
        print(f"  {mag:<6} → Acc: {mm['accuracy']*100:.2f}%  F1: {mm['f1_macro']*100:.2f}%  AUC: {mm['auc_roc']*100:.2f}%")

print()
print('='*60)
print('  MULTICLASS RESULTS (8 tumor subtypes)')
print('='*60)
for model_name in ['cnn_vit', 'efficientnet_vit']:
    path = os.path.join(RESULTS_MULTICLASS, f'{model_name}_metrics.json')
    if not os.path.exists(path):
        print(f'  {model_name}: not found'); continue
    with open(path) as f:
        m = json.load(f)
    o = m['overall']
    print(f'\n  {model_name.upper()}')
    print(f"  Overall  → Acc: {o['accuracy']*100:.2f}%  F1: {o['f1_macro']*100:.2f}%  AUC: {o['auc_roc']*100:.2f}%")
    for mag, mm in m.get('per_magnification', {}).items():
        print(f"  {mag:<6} → Acc: {mm['accuracy']*100:.2f}%  F1: {mm['f1_macro']*100:.2f}%  AUC: {mm['auc_roc']*100:.2f}%")

print(f'\n✓ All results saved to Drive.')